# Transformer From Scratch

In the lecturer, we discussed how the [Transformer](https://arxiv.org/abs/1706.03762) looks like. Let's implement it.

![img](https://jalammar.github.io/images/t/transformer_resideual_layer_norm_3.png)

We will start with positional encoding. There are many ways of doing it. A simple one is just to use a `torch.nn.Embedding`, however, the original approach used a sine-based equation:

$$
PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d}}\right), \qquad PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d}}\right),
$$
where $i$ is the dimension index and $pos$ is the position index.

Since all dimensions have different frequencies, some change slowly (can distinguish far positions), some fast (can distinguish close positions), and we cover different range sesitivity.

Assuming $d$ is even, having $\sin$ and $\cos$ allows:
$$
PE(pos+k) = \text{Rotation}(k)PE(pos)
$$

Indeed,
$$
\delta = \frac{k}{10000^{2i/d}}
$$

Then

$$
v(pos + k)
=
\begin{bmatrix}
\sin(\theta + \delta) \\
\cos(\theta + \delta)
\end{bmatrix}
$$

$$
\sin(\theta + \delta)
=
\sin\theta \cos\delta
+
\cos\theta \sin\delta
$$

$$
\cos(\theta + \delta)
=
\cos\theta \cos\delta
-
\sin\theta \sin\delta
$$

We get

$$
v(pos + k)
=
\begin{bmatrix}
\cos\delta & \sin\delta \\
-\sin\delta & \cos\delta
\end{bmatrix}
\begin{bmatrix}
\sin\theta \\
\cos\theta
\end{bmatrix}
$$

Note that in self-attention, we compute

$$
A = \text{Softmax}\left(\frac{QK^T}{\sqrt{d}}\right)
$$

Such PE allows $QK^T$ to use information about relative position. Indeed, given
$$
\theta_1 = \frac{pos_1}{10000^{2i/d}},
\qquad
\theta_2 = \frac{pos_2}{10000^{2i/d}}
$$
The positional vector (for that frequency pair) is
$$
v(pos) =
\begin{bmatrix}
\sin\theta \\
\cos\theta
\end{bmatrix}
$$
Now compute the dot product:
$$
v(pos_1)^\top v(pos_2)
=
\sin\theta_1 \sin\theta_2
+
\cos\theta_1 \cos\theta_2
$$
Hence,
$$
v(pos_1)^\top v(pos_2)
=
\cos(\theta_1 - \theta_2)
=
\cos\left(
\frac{pos_1 - pos_2}{10000^{2i/d}}
\right)
$$


Now, let's code it in `PyTorch`.

Now, let's work on Attention:

$$
A = \text{Softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V
$$

We will directly code multi-head attention. In it, the hidden dimension $d$ is split into $n$ equal-size partitions, where $n$ is the number of heads.
$$
O = W_O\text{Concat}[A_1, ..., A_n]
$$

Now we are left with coding FeedForward and Encoder/Decoder layers.

Feedforward is just $FC_2(\text{activation}(FC_1(x)))$

Finally, we need to combine all of this into a single architecture

We do not provide code for padding masking and casual masking here to keep it for the homework. However, let's discuss the purpose:

Padding mask: avoid calculating attention and using meaningless PAD tokens.

Causal mask (**Only in autoregressive setup, i.e. Decoder**): ensure that token $i$ sees only tokens $j \le i$ and does not look into the future. This prevents the Transformer to cheat and learn mapping from the future token, which would not be possible during inference.

# HuggingFace

[HuggingFace](https://huggingface.co/) is a great set of tools useful for Deep Learning applications and Research, especially Transformers-related.

Features include (not exhaustive list):

* [datasets](https://huggingface.co/docs/datasets/index) library: nice package for working with different types of data. Enables multi-thread pre-processing, filtering, sorting, caching, saving, etc.. It also allows to preview data [online](https://huggingface.co/datasets)

* [transformers](https://huggingface.co/docs/transformers/index) library: framework for coding, running, and training models, mostly Transformer-based.

* [huggingface hub](https://huggingface.co/docs/hub/index): download\upload modules for sharing weights/datasets/etc.

* [accelerate](https://huggingface.co/docs/accelerate/index) library: easy handling of multi-GPU training and precision modifications.

* [PEFT](https://huggingface.co/docs/peft/index) library: additional tools to efficiently fine-tune large models

Let's walk through some things we can do with HF. (See [HF Course](https://huggingface.co/learn/llm-course) and other guides for more details).

For example, it already has some pre-defined pipelines you can plug-and-play with:

![img](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline.svg)

In [ ]:
from transformers import pipeline

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
classifier = pipeline("sentiment-analysis", model=checkpoint)
raw_inputs = [
    "I love deep learning in Audio.",
    "I hate the first homework!",
]
classifier(
    raw_inputs
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9997804760932922},
 {'label': 'NEGATIVE', 'score': 0.9995282888412476}]

Let's walk through each elements of the pipeline step-by-step.

First, we need to somehow process string text. As we discussed in the lecture, we use a tokenizer for this. It has a pre-defined set of units (tokens) and splits text into it. HuggingFace has built-in Tokenizer modules with the possibility to download pre-defined tokeniezers.

In [ ]:
from transformers import AutoTokenizer # automatically gets the tokenizer class name

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Let's look what's inside

In [ ]:
tokenizer

BertTokenizer(name_or_path='distilbert-base-uncased-finetuned-sst-2-english', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

We see that there are special tokens, such as `[UNK]` and the whole vocabulary is quite large. Let's have a look at some elements

In [ ]:
list(tokenizer.vocab.items())[:10]

[('perception', 10617),
 ('##uin', 20023),
 ('shift', 5670),
 ('punish', 16385),
 ('rumours', 19200),
 ('schedule', 6134),
 ('metropolis', 18236),
 ('##till', 28345),
 ('killer', 6359),
 ('##omorphic', 28503)]

Here `##` indicates that this token should be concatenated without a space. Having sub-words reveals that this is a subwor-tokenization (like BPE) tokenizer.

In [ ]:
print(raw_inputs)
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

['I love deep learning in Audio.', 'I hate the first homework!']
{'input_ids': tensor([[  101,  1045,  2293,  2784,  4083,  1999,  5746,  1012,   102],
        [  101,  1045,  5223,  1996,  2034, 19453,   999,   102,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0]])}


As sequences are of different length, the shorter one had to be padded with a PAD token. We can try to decode back

In [ ]:
tokenizer.decode(inputs.input_ids)

['[CLS] i love deep learning in audio. [SEP]',
 '[CLS] i hate the first homework! [SEP] [PAD]']

To remove special tokens, we need to explicitly request it

In [ ]:
tokenizer.decode(inputs.input_ids, skip_special_tokens=True)

['i love deep learning in audio.', 'i hate the first homework!']

Similartly, we can load models from the [Model Hub](https://huggingface.co/models)

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.weight | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
model

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

Actually, it is just a special `PyTorch` subclass, so we can just pass tokenized inputs and we get the output, like in `PyTorch`.

In [ ]:
outputs = model(**inputs)
outputs

BaseModelOutput(last_hidden_state=tensor([[[ 0.3927,  0.0456,  0.1050,  ...,  0.4011,  0.8904, -0.3763],
         [ 0.7617,  0.1591, -0.1150,  ...,  0.3993,  0.8889, -0.2825],
         [ 0.7384,  0.3026,  0.1868,  ...,  0.3300,  0.8083, -0.1789],
         ...,
         [ 0.2896,  0.1124,  0.2883,  ...,  0.4179,  0.6507, -0.5020],
         [ 0.6298, -0.0672,  0.1469,  ...,  0.5632,  0.8766, -0.6742],
         [ 1.0683,  0.0872,  0.3574,  ...,  0.8427,  0.5347, -0.8026]],

        [[-0.1777,  0.9128, -0.1175,  ..., -0.0013, -0.8777, -0.1410],
         [-0.2240,  1.0403, -0.1186,  ..., -0.2052, -0.7015,  0.2762],
         [-0.1363,  1.0532, -0.0086,  ..., -0.0108, -0.7053,  0.0123],
         ...,
         [ 0.0351,  1.0268, -0.1966,  ..., -0.1107, -0.8206, -0.2553],
         [ 0.1890,  0.6743, -0.2488,  ..., -0.1089, -0.6030, -0.0882],
         [-0.1344,  1.1360, -0.0085,  ..., -0.0090, -0.6558, -0.1871]]],
       grad_fn=<NativeLayerNormBackward0>), hidden_states=None, attentions=None)

Some base model checkpoints only return hidden states, not the model with a head to do classification, generation, etc.

![img](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/transformer_and_head.svg)

If we write our own code, we can add layers on top ourselves. If we want to stay within HF framework, we will need to stay within the HF model design (see [BERT Modeling](https://github.com/huggingface/transformers/blob/main/src/transformers/models/bert/modeling_bert.py#L549) for an example.). Most models also support basic set of heads.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-4.0635,  4.3604],
        [ 4.2534, -3.4052]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [ ]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions)

tensor([[2.1951e-04, 9.9978e-01],
        [9.9953e-01, 4.7173e-04]], grad_fn=<SoftmaxBackward0>)


We can access model config to convert scores to labels

In [ ]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

## Datasets

First, there are many datasets available, which you can easily download and use:

In [ ]:
import datasets

ds = datasets.load_dataset("zalando-datasets/fashion_mnist")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

fashion_mnist/train-00000-of-00001.parqu(…):   0%|          | 0.00/30.9M [00:00<?, ?B/s]

fashion_mnist/test-00000-of-00001.parque(…):   0%|          | 0.00/5.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 60000
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

We can look at a particular split and element:

In [ ]:
train_ds = ds["train"]
elem = train_ds[0]
elem

{'image': <PIL.PngImagePlugin.PngImageFile image mode=L size=28x28>,
 'label': 9}

Each element is a dict: column name -> value

In [ ]:
elem["image"]

One can also create a HF dataset from raw files. For example, a [Drug Review](https://archive.ics.uci.edu/dataset/461/drug+review+dataset+druglib+com) dataset.

In [ ]:
!wget "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"
!unzip drugsCom_raw.zip

--2026-02-21 02:02:42--  https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘drugsCom_raw.zip’

drugsCom_raw.zip        [        <=>         ]  41.00M  26.0MB/s    in 1.6s    

2026-02-21 02:02:44 (26.0 MB/s) - ‘drugsCom_raw.zip’ saved [42989872]

Archive:  drugsCom_raw.zip
  inflating: drugsComTest_raw.tsv    
  inflating: drugsComTrain_raw.tsv   


In [ ]:
from datasets import load_dataset

data_files = {"train": "drugsComTrain_raw.tsv", "test": "drugsComTest_raw.tsv"}
# \t is the tab character in Python
drug_dataset = load_dataset("csv", data_files=data_files, delimiter="\t")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [ ]:
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

One we processed it, we can save it on disk or upload to huggingface.

In [ ]:
drug_dataset.save_to_disk("dataset_example")

Saving the dataset (0/1 shards):   0%|          | 0/161297 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/53766 [00:00<?, ? examples/s]

In [ ]:
!ls dataset_example

dataset_dict.json  test  train


In [ ]:
!cat dataset_example/dataset_dict.json

{"splits": ["train", "test"]}

In [ ]:
!ls dataset_example/train

data-00000-of-00001.arrow  dataset_info.json  state.json


In [ ]:
!cat dataset_example/train/dataset_info.json

{
  "builder_name": "csv",
  "citation": "",
  "config_name": "default",
  "dataset_name": "csv",
  "dataset_size": 116144739,
  "description": "",
  "download_checksums": {
    "/content/drugsComTrain_raw.tsv": {
      "num_bytes": 84289175,
      "checksum": null
    },
    "/content/drugsComTest_raw.tsv": {
      "num_bytes": 28071166,
      "checksum": null
    }
  },
  "download_size": 112360341,
  "features": {
    "Unnamed: 0": {
      "dtype": "int64",
      "_type": "Value"
    },
    "drugName": {
      "dtype": "string",
      "_type": "Value"
    },
    "condition": {
      "dtype": "string",
      "_type": "Value"
    },
    "review": {
      "dtype": "string",
      "_type": "Value"
    },
    "rating": {
      "dtype": "float64",
      "_type": "Value"
    },
    "date": {
      "dtype": "string",
      "_type": "Value"
    },
    "usefulCount": {
      "dtype": "int64",
      "_type": "Value"
    }
  },
  "homepage": "",
  "license": "",
  "size_in_bytes": 228505080,
  

`dataset` allow you to apply many operation on the dataset. Such a select a subset of indexes using `select` or shuffle via `shuffle`

In [ ]:
drug_sample = drug_dataset["train"].shuffle(seed=42).select(range(1000))
# Peek at the first few examples
drug_sample[:3]

{'Unnamed: 0': [87571, 178045, 80482],
 'drugName': ['Naproxen', 'Duloxetine', 'Mobic'],
 'condition': ['Gout, Acute', 'ibromyalgia', 'Inflammatory Conditions'],
 'review': ['"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
  '"I have taken Cymbalta for about a year and a half for fibromyalgia pain. It is great\r\nas a pain reducer and an anti-depressant, however, the side effects outweighed \r\nany benefit I got from it. I had trouble with restlessness, being tired constantly,\r\ndizziness, dry mouth, numbness and tingling in my feet, and horrible sweating. I am\r\nbeing weaned off of it now. Went from 60 mg to 30mg and now to 15 mg. I will be\r\noff completely in about a week. The fibro pain is coming back, but I would rather deal with it than the side effects."',
  '"I have been taking Mobic for over a year with no side effects other than 

We see some issues in the dataset and can quickly fix them directly using other HF dataset operations:

1. The Unnamed: 0 column looks suspiciously like an anonymized ID for each patient.
2. The condition column includes a mix of uppercase and lowercase labels.
3. The reviews are of varying length and contain a mix of Python line separators (\r\n) as well as HTML character codes like &\#039;.

In [ ]:
# Rename columns
drug_dataset = drug_dataset.rename_column(
    original_column_name="Unnamed: 0", new_column_name="patient_id"
)
drug_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

In [ ]:
# convert to lowercase
def lowercase_condition(example):
    return {"condition_new": example["condition"].lower()}


drug_dataset.map(lowercase_condition)

Map:   0%|          | 0/161297 [00:00<?, ? examples/s]

AttributeError: 'NoneType' object has no attribute 'lower'

We need to filter NaNs first:

In [ ]:
# use filter method
drug_dataset = drug_dataset.filter(lambda x: x["condition"] is not None)

Filter:   0%|          | 0/161297 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53766 [00:00<?, ? examples/s]

In [ ]:
drug_dataset.map(lowercase_condition)

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 160398
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53471
    })
})

One could parallize it or batchify for speed-up

In [ ]:
drug_dataset.map(lowercase_condition, num_proc=4)

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 160398
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53471
    })
})

Note that we got the result without any processing for the second time. This is because HF proceses operations and just loads the result from cache.

Let's use another function for demonstration

In [ ]:
def uppercase_condition(example):
    return {"condition": example["condition"].upper()}

In [ ]:
drug_dataset.map(uppercase_condition, num_proc=4)

Map (num_proc=4):   0%|          | 0/160398 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/53471 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 160398
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53471
    })
})

And see again that is is cached

In [ ]:
drug_dataset.map(uppercase_condition, num_proc=4)

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 160398
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53471
    })
})

We can also add new columns and do much more using `map` or specialized methods

In [ ]:
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}

In [ ]:
drug_dataset = drug_dataset.map(compute_review_length)
# Inspect the first training example
drug_dataset["train"][0]

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

{'patient_id': 206461,
 'drugName': 'Valsartan',
 'condition': 'Left Ventricular Dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'review_length': 17}

## Training with HF Trainer

While we can still use `PyTorch` training loop as in previous seminars, it might be more convenient to work inside the HF framework.

Many Sequence Modeling tasks are pre-defined in HF. The trainer loop automatically logs everything to `WandB` or whatever tool you use. It can do distributed (multi-GPU) training, save and upload all checkpoints, etc., including shortening the overall amount of code

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

`Trainer` class does all the training, evaluation, and saving. To control it, we need `TrainingArguments`.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments("test-trainer")

In [ ]:
training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.NO,
eval_use_gather_object=False,

Let's use some model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Now we can train it using `Trainer` class.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

# vLLM and OpenAI client

In [ ]:
!pip install -q vllm==0.11.0 "transformers>=4.51.0,<5" openai

In [ ]:
!vllm serve Qwen/Qwen3-0.6B \
    --port 8000 \
    --host 127.0.0.1 \
    > vllm.log 2>&1 &

In [ ]:
!cat vllm.log

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",
    api_key="EMPTY",
)

# (Optional) sanity check that the server responds
print(client.models.list())

resp = client.chat.completions.create(
    model="Qwen/Qwen3-0.6B",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Write a 5-line poem about GPUs."},
    ],
    temperature=0.7,
    max_tokens=120,
)

print(resp.choices[0].message.content)